In [ ]:
from model import *
from utils import *

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt 
from PIL import Image 
import os 

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

train_data = load_data(r'\train\train\train.npz')
X, y, _ = prepare_train_data(train_data)

DATASET_SIZE = len(X)
# Split train/val
# This is done only to measure generalization capabilities, you don't have to
# use a validation set (though we encourage this)
n_train = int(0.9 * len(X))
TRAIN_SPLIT = torch.zeros(len(X), dtype=torch.bool)
TRAIN_SPLIT[:n_train] = 1
X_train, X_val = X[TRAIN_SPLIT], X[~TRAIN_SPLIT]
y_train, y_val = y[TRAIN_SPLIT], y[~TRAIN_SPLIT]


train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256)

cuda
(125000,)
Train data: 125000 captions, 125000 images


# Best ExpandCompressMLP Model


The next training is created using the parameter got from a previous grid-search using Optuna, focussing on learning rate and weight decay, and we changed a little the temperature without touching the general structure of the model. This was the best ExpandCompressMLP we got by Public Score on Kaggle.

In [ ]:
bottleneck_model =  ExpandCompressMLP(
        input_dim=1024,
        hidden_dim=1408,
        bottleneck_dim=1024,
        output_dim=1536,
        dropout=0.4,
        use_residual=False
    ).to(device)


# To Train the model, uncomment the following block, otherwise there is the next cell with the code to load the trained model
# THERE IS NO SEED THAT ENSURE THE SAME OUTPUT MODEL, sorry :( 
'''   
trained_model, _ = train_model(
        model,
        train_loader,
        val_loader,
        device=device,
        lr=0.00017797872654091814,
        weight_decay=0.000020668777307711872,
        temperature=0.15,
        epochs=50,
        patience=7,
        use_amp=False,
        use_scheduler=True)
'''

'\ntrained_model, _ = train_model(\n        model,\n        train_loader,\n        val_loader,\n        device=device,\n        lr=0.00017797872654091814,\n        weight_decay=0.000020668777307711872,\n        temperature=0.15,\n        epochs=50,\n        patience=7,\n        use_amp=False,\n        use_scheduler=True)\n'

In [ ]:
bottleneck_model.load_state_dict(torch.load(r"\Models\Second_submission\Second_model.pth"))

<All keys matched successfully>

In [ ]:
res = evaluate_model(bottleneck_model, val_loader, device)


Risultati complessivi su 12500 campioni:
Top-1 accuracy : 19.86%
Top-5 accuracy : 93.50%
Top-10 accuracy: 98.32%
Rank medio     : 3.48
MRR            : 0.4449


In [ ]:
test_data = load_data(r'\test\test\test.clean.npz')
test_embds = torch.from_numpy(test_data['captions/embeddings']).float()
caption_ids = test_data['captions/ids']

pred_embds = predict_in_batches(bottleneck_model, test_embds, device, batch_size=64)

submission = generate_submission(caption_ids, pred_embds, 'submission_best_model_for_now_1408_1024_4_1_new.csv')

Generating submission file...
✓ Saved submission to submission_bottleneck_crazy2.csv
